In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping

from imblearn.over_sampling import SMOTE #for handling imbalanced images dataset

import pathlib


import cv2

In [2]:
datadir=pathlib.Path("DataSet_Major_Project/Multiple_Sclerosis/")
datadir

WindowsPath('DataSet_Major_Project/Multiple_Sclerosis')

In [3]:
list(datadir.glob("*/*.png"))[:5]

[WindowsPath('DataSet_Major_Project/Multiple_Sclerosis/Control-Axial/C-A (1).png'),
 WindowsPath('DataSet_Major_Project/Multiple_Sclerosis/Control-Axial/C-A (10).png'),
 WindowsPath('DataSet_Major_Project/Multiple_Sclerosis/Control-Axial/C-A (100).png'),
 WindowsPath('DataSet_Major_Project/Multiple_Sclerosis/Control-Axial/C-A (1000).png'),
 WindowsPath('DataSet_Major_Project/Multiple_Sclerosis/Control-Axial/C-A (1001).png')]


#### Create list of files in each MS sub folder

In [4]:
MS_images_dict={
    'Control_Axial':list(datadir.glob("Control-Axial/*")),
    'Control_Sagittal':list(datadir.glob("Control-Sagittal/*")),
    'MS_Axial':list(datadir.glob("MS-Axial/*")),
    'MS_Sagittal':list(datadir.glob("MS-Sagittal/*"))
}

In [5]:
MS_images_dict['MS_Axial'][:10]

[WindowsPath('DataSet_Major_Project/Multiple_Sclerosis/MS-Axial/MS-A (1).png'),
 WindowsPath('DataSet_Major_Project/Multiple_Sclerosis/MS-Axial/MS-A (10).png'),
 WindowsPath('DataSet_Major_Project/Multiple_Sclerosis/MS-Axial/MS-A (100).png'),
 WindowsPath('DataSet_Major_Project/Multiple_Sclerosis/MS-Axial/MS-A (101).png'),
 WindowsPath('DataSet_Major_Project/Multiple_Sclerosis/MS-Axial/MS-A (102).png'),
 WindowsPath('DataSet_Major_Project/Multiple_Sclerosis/MS-Axial/MS-A (103).png'),
 WindowsPath('DataSet_Major_Project/Multiple_Sclerosis/MS-Axial/MS-A (104).png'),
 WindowsPath('DataSet_Major_Project/Multiple_Sclerosis/MS-Axial/MS-A (105).png'),
 WindowsPath('DataSet_Major_Project/Multiple_Sclerosis/MS-Axial/MS-A (106).png'),
 WindowsPath('DataSet_Major_Project/Multiple_Sclerosis/MS-Axial/MS-A (107).png')]

In [6]:
MS_label_dict={
    'Control_Axial':0,
    'Control_Sagittal':1,
    'MS_Axial':2,
    'MS_Sagittal':3
}


### Converting image to numpy array

In [7]:
img=cv2.imread(str(MS_images_dict['Control_Axial'][2]))
img

array([[[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       ...,

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]]], dtype=uint8)

In [8]:
img.shape

(569, 1158, 3)

##### resize image to a uniform size

In [9]:
cv2.resize(img,(180,180)).shape

(180, 180, 3)

## Preparing data for model

In [ ]:
X,y=[],[]

for msCategoryName,msCategoryImages in MS_images_dict.items():
    for image in msCategoryImages:
        img=cv2.imread(str(image))
        resized_img=cv2.resize(img,(180,180))
        X.append(resized_img)
        y.append(MS_label_dict[msCategoryName])

In [ ]:
X=np.array(X)
y=np.array(y)

In [ ]:
len(X)

# Checking and handling imbalanced images count

In [ ]:
y1=pd.DataFrame(y,columns=['Classes'])
y1.value_counts()

In [ ]:
X.shape

In [ ]:
X1 = np.reshape(X, (X.shape[0], -1))
X1.shape

In [ ]:

smote = SMOTE(sampling_strategy='auto')
X_sm, y_sm = smote.fit_resample(X1, y1)

X_sm = np.reshape(X_sm, (X_sm.shape[0], 180, 180, 3))

In [ ]:
X_sm.shape

In [ ]:
y_sm.shape

In [ ]:
y_sm.value_counts()

## Data is balanced now(see above)

In [ ]:
X=X_sm
y=y_sm['Classes'].values

print("X_Shape = "+ str(X.shape))
print("y_Shape = "+ str(y.shape))

In [ ]:
def getCategoryName(n):
    for x,y in MS_label_dict.items():
        if y==n:
            return x


plt.figure(figsize=(20,10))
for n,i in enumerate(list(np.random.randint(0,len(X),8))):
    plt.subplot(2,4,n+1)
    plt.imshow(X[i])
    plt.axis('Off')
    plt.title(getCategoryName(y[i]))

#### Scalling b/w 0 and 1

In [ ]:
X=X/255;

## Splitting data

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.shape[0]

In [ ]:
X_test.shape[0]

## Creating CNN

In [ ]:
model=Sequential([
    layers.Conv2D(filters=32,kernel_size=(3,3),activation='relu',input_shape=(180,180,3)),
    layers.MaxPool2D((2,2)),

    # layers.Conv2D(filters=32,kernal_size=(3,3),activation='relu',input_shape=(180,180,3)),
    # layers.MaxPool2D((2,2))

    #flattening and getting result
    layers.Flatten(),
    layers.Dense(64,activation='relu'),
    layers.Dense(4,activation='sigmoid') #4 classes   
])


model.compile(optimizer='adam',
               loss='sparse_categorical_crossentropy',
               metrics=['accuracy'])


early_stopping=EarlyStopping(monitor='val_loss',#preventing overfitting
                             patience=3,
                             restore_best_weights=True)

history=model.fit(X_train,y_train,
                  epochs=15,
                  validation_data=(X_test,y_test),
                  callbacks=[early_stopping])

## Accuracy and Loss Graph

In [ ]:

#Accuracy garph
plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(["Train",'Validation'],loc='upper left')

#loss graph
plt.subplot(1,2,2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(["Train",'Validation'],loc='upper left')
plt.tight_layout()
plt.show


In [ ]:
model.evaluate(X_test,y_test)

In [ ]:
predictions=model.predict(X_test)
predictions

In [ ]:
preds=[np.argmax(i) for i in predictions]

In [ ]:
str(preds)

In [ ]:
from sklearn.metrics import confusion_matrix,classification_report

print("classification report : \n",classification_report(y_test,preds))

In [ ]:
cm=tf.math.confusion_matrix(labels=y_test,predictions=preds)

import seaborn as sns
plt.figure(figsize=(10,7))
sns.heatmap(cm,annot=True,fmt='d')
plt.xlabel('Predicted')
plt.ylabel('Truth')